# Text to image: a recipe, not a pipeline task

[Back to the introduction](ch01_demo.ipynb). Run in order, in a fresh kernel, in a browser with a **hardware GPU** (recent Chrome or Edge, or Safari 26). This notebook generates a 512 x 512 image from a prompt with SD-Turbo, in one denoising step, entirely in the browser. It builds on `OnnxModel` (bare ONNX files, NumPy in and out): there is no `text-to-image` pipeline task, because the pieces differ for every model family (see [the model guide](https://github.com/decitre/lmlite/blob/main/docs/MODELS.md#text-to-image-a-recipe-not-a-pipeline-task) for the numbers behind that decision).

**Read this first**

- **Licence: non-commercial research only.** SD-Turbo is under the Stability AI Non-Commercial Research Community License. It forbids production use and offering the model through a hosted service. This notebook downloads the weights from the Hugging Face Hub into your browser; do not bundle them into a site you publish.
- **Download: 2.4 GB** (a 1.7 GB UNet, a 0.7 GB text encoder, a 0.1 GB decoder), on every new kernel: the notebook does not keep them in the browser cache (a 1.7 GB file did not fit in the cache of the test browser).
- **Memory: about 7 GB** for the whole browser, measured on an Apple M-series Mac with 18 GB.
- **It needs WebGPU with float16 shaders and 1.7 GB buffers.** WebAssembly cannot allocate the UNet, and a software WebGPU adapter has no float16 support. The next cell checks the adapter.

In [ ]:
import json
import time

import js
import numpy as np
from IPython.display import Image, display
from PIL import Image as PilImage

from lmlite import LLM, OnnxModel

gpu = js.eval("""(async () => {
  const adapter = await navigator.gpu?.requestAdapter({powerPreference: 'high-performance'});
  if (!adapter) return JSON.stringify(null);
  const info = adapter.info ?? {};
  return JSON.stringify({vendor: info.vendor, architecture: info.architecture,
                         fallback: info.isFallbackAdapter ?? adapter.isFallbackAdapter ?? null,
                         shader_f16: adapter.features.has('shader-f16'), max_buffer_bytes: adapter.limits.maxBufferSize});
})""")
adapter = json.loads(str(await gpu()))
print("WebGPU adapter:", adapter)
UNET_BYTES = 1_733_430_199
if adapter is None or adapter["fallback"] is True or not adapter["shader_f16"] or adapter["max_buffer_bytes"] < UNET_BYTES:
    raise RuntimeError("This needs a hardware WebGPU adapter with float16 shaders and buffers of at least 1.7 GB.")

## Load the three models

The text encoder turns the prompt into 77 vectors, the UNet predicts the noise in a 4 x 64 x 64 latent image, and the decoder turns the latent into pixels. `OnnxModel.load()` takes a URL; `cache=False` skips the Cache API, which cannot hold the UNet. Each model declares its input dtypes, and `cast` converts to them.

In [ ]:
BASE = "https://huggingface.co/schmuell/sd-turbo-ort-web/resolve/main"
load_seconds = {}


async def load(part):
    start = time.perf_counter()
    model = await OnnxModel.load(f"{BASE}/{part}/model.onnx", backend="webgpu", cache=False, licence="Stability AI Non-Commercial Research Community License")
    load_seconds[part] = round(time.perf_counter() - start, 1)
    return model


tokenizer = await LLM.tokenizer("Xenova/clip-vit-base-patch32")  # the CLIP vocabulary SD-Turbo's text encoder was trained with
text_encoder = await load("text_encoder")
unet = await load("unet")
decoder = await load("vae_decoder")
print("load seconds:", load_seconds)
for name, model in [("text_encoder", text_encoder), ("unet", unet), ("vae_decoder", decoder)]:
    print(name, [(i["name"], i["dtype"]) for i in model.inputs], "->", [o["name"] for o in model.outputs])

## One denoising step

SD-Turbo is distilled to produce an image in one step. Its scheduler is Euler with `scaled_linear` betas (0.00085 to 0.012 over 1,000 steps) and `trailing` spacing, so the single timestep is 999. The start latent is Gaussian noise times the largest sigma (about 14.6); the UNet sees it scaled by `1 / sqrt(sigma**2 + 1)`; one Euler step to sigma 0 gives `latents - sigma * noise_prediction`; the decoder expects the latent divided by 0.18215. The prompt is padded to 77 tokens with the `!` token (id 0).

In [ ]:
def cast(model, arrays):
    """Each array in the dtype the model declares for that input."""
    dtypes = {i["name"]: i["dtype"] for i in model.inputs}
    return {name: array.astype(dtypes[name]) for name, array in arrays.items()}


def first_output(outputs, model):
    return outputs[model.outputs[0]["name"]]


async def generate(prompt, seed=0):
    """A 512 x 512 PNG for the prompt, and the seconds each stage took."""
    start = time.perf_counter()
    ids = tokenizer.encode(prompt, add_special_tokens=True)[:77]
    ids = ids + [0] * (77 - len(ids))
    hidden = first_output(await text_encoder.run(cast(text_encoder, {"input_ids": np.array([ids])})), text_encoder)
    after_text = time.perf_counter()

    betas = np.linspace(0.00085**0.5, 0.012**0.5, 1000, dtype=np.float64) ** 2
    alphas = np.cumprod(1 - betas)
    sigma = float(np.sqrt((1 - alphas[999]) / alphas[999]))
    latents = np.random.default_rng(seed).standard_normal((1, 4, 64, 64)).astype(np.float32) * sigma
    noise = first_output(
        await unet.run(cast(unet, {
            "sample": latents / np.sqrt(sigma**2 + 1),
            "timestep": np.array([999]),
            "encoder_hidden_states": hidden,
        })),
        unet,
    )
    after_unet = time.perf_counter()

    denoised = latents - sigma * noise.astype(np.float32)
    image = first_output(await decoder.run(cast(decoder, {"latent_sample": denoised / 0.18215})), decoder)
    pixels = ((image[0].astype(np.float32) / 2 + 0.5).clip(0, 1) * 255).round().astype(np.uint8).transpose(1, 2, 0)
    done = time.perf_counter()

    import io

    png = io.BytesIO()
    PilImage.fromarray(pixels).save(png, format="PNG")
    seconds = {"text encoder": after_text - start, "unet": after_unet - after_text, "decoder": done - after_unet, "total": done - start}
    return png.getvalue(), {stage: round(value, 2) for stage, value in seconds.items()}

## Generate

Change the prompt or the seed and run the cell again. The first image is slower than the next ones, because the GPU compiles its shaders on first use.

In [ ]:
png, seconds = await generate("a photo of an astronaut riding a horse on mars", seed=7)
print("seconds:", seconds)
display(Image(data=png, width=384))

heap = js.eval("""(async () => {
  const memory = performance.memory;  // Chromium only: the JavaScript heap, not the GPU or WebAssembly memory
  return JSON.stringify(memory ? {used_mb: Math.round(memory.usedJSHeapSize / 1e6), limit_mb: Math.round(memory.jsHeapSizeLimit / 1e6)} : null);
})""")
print("JavaScript heap:", json.loads(str(await heap())) or "not available in this browser (Chromium only)")

## Try your own prompts

Edit the text and press **Generate**. **Another image** keeps the text and draws a new random seed; the same text and seed always give the same image. **Save to PNG** writes the picture to the notebook's file browser and offers it as a download. The widgets below call `generate()` from above, so run that cell first and run this one before the last cell, which releases the models.

In [ ]:
import asyncio
import base64
import random

import ipywidgets as widgets

prompt = widgets.Textarea(value="a photo of an astronaut riding a horse on mars", layout=widgets.Layout(width="100%"))
seed = widgets.BoundedIntText(value=7, min=0, max=2**31 - 1, description="Seed")
generate_button = widgets.Button(description="Generate", button_style="primary")
another = widgets.Button(description="Another image")
save = widgets.Button(description="Save to PNG", disabled=True)
status = widgets.Label("Ready.")
picture = widgets.Image(format="png", width=384)
link = widgets.HTML()
last = {}


async def run():
    text = prompt.value.strip()
    if not text:
        status.value = "Enter a prompt first."
        return
    generate_button.disabled = another.disabled = save.disabled = True
    status.value = "Generating..."
    try:
        last["png"], last["seed"] = None, seed.value
        png, seconds = await generate(text, seed=seed.value)
        last["png"] = png
        picture.value = png
        link.value = ""
        status.value = f"Done in {seconds['total']:.1f} s."
    except Exception as error:
        status.value = f"Failed: {error}"
    finally:
        generate_button.disabled = another.disabled = False
        save.disabled = not last.get("png")


def on_another(_):
    seed.value = random.randrange(2**31)
    asyncio.create_task(run())


def on_save(_):
    name = f"image-{last['seed']}.png"
    with open(name, "wb") as file:
        file.write(last["png"])
    data = base64.b64encode(last["png"]).decode()
    link.value = f'Saved <code>{name}</code> in the file browser. <a download="{name}" href="data:image/png;base64,{data}">Download it</a>'


generate_button.on_click(lambda _: asyncio.create_task(run()))
another.on_click(on_another)
save.on_click(on_save)
widgets.VBox([prompt, widgets.HBox([seed, generate_button, another, save]), status, picture, link])

## What it costs, and why it is a recipe

Measured on an Apple M-series Mac (18 GB), Chromium on the Metal GPU, Pyodide 0.29, with the weights served from the same machine (ranges over three runs):

| | |
|:--|:--|
| Load (UNet, text encoder, decoder) | 7.0 to 8.2 s + 2.1 to 2.6 s + 0.4 to 0.5 s, after a 7 minute download of 2.4 GB |
| First image | 2.8 to 4.3 s over three runs |
| Next images | 2.5 to 3.4 s |
| Whole-browser memory, peak | about 6.6 to 7.2 GB (1.8 to 1.9 GB of it the JavaScript heap, of a 3.8 GB limit) |
| WebAssembly backend | fails: `failed to allocate a buffer of size 1733430199` |
| Software WebGPU adapter | fails: `Program Transpose requires f16 but the device does not support it` |

That is usable on a recent laptop GPU and out of reach on most phones, tablets and machines without a float16 GPU, with a download that repeats each session, and under a licence that rules out shipping it. Every model family also needs its own tokenizer padding, scheduler and decoder scaling, so a `text-to-image` task would promise more than lmlite can check. The recipe stays here, where the limits are visible.

In [ ]:
await text_encoder.dispose()
await unet.dispose()
await decoder.dispose()
tokenizer.close()